<a href="https://colab.research.google.com/github/mahmoud-mos/my-ml-internship/blob/main/my-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoud-mos/my-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Ranking/Scoring. This task is structured as a ranking problem because our objective is to prioritize content items based on their likelihood of needing a refresh, enabling efficient allocation of human review capacity.

In [ ]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(f"Total content items queued for priority ranking: {len(df):,}")

Total content items queued for priority ranking: 100


In [ ]:
import pandas as pd
import os

# Define the path for the dummy CSV file
file_path = 'data/raw/content_refresh_anonymized.csv'

# Create directories if they don't exist
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Create a dummy DataFrame with required columns
dummy_data = {
    'content_hash_id': [f'hash_{i}' for i in range(100)],
    'trend_direction': ['down'] * 20 + ['up'] * 80,
    'days_since_last_update': [i * 5 for i in range(100)],
    'impressions_90d': [i * 1000 for i in range(100)],
    'clicks_90d': [i * 50 for i in range(100)]
}
dummy_df = pd.DataFrame(dummy_data)

# Save the dummy DataFrame to a CSV file
dummy_df.to_csv(file_path, index=False)

print(f"Dummy file '{file_path}' created with {len(dummy_df)} rows.")
print("Please re-run the notebook from the beginning after this cell execution to ensure all dependencies are met.")


Dummy file 'data/raw/content_refresh_anonymized.csv' created with 100 rows.
Please re-run the notebook from the beginning after this cell execution to ensure all dependencies are met.


## 2. Target or proxy

Binary classification proxy. We define a page as actively decaying—and thus a candidate for refresh—if its `trend_direction` is observed as `"down"`. This proxy serves as our ground-truth label for modeling.

In [ ]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print("Target Class Distribution (trend_direction):")
print(df['trend_direction'].value_counts(normalize=True))

Target Class Distribution (trend_direction):
trend_direction
up      0.8
down    0.2
Name: proportion, dtype: float64


## 3. Success metric

Precision @50 and ROC AUC. We prioritize **Precision @50** to ensure that the top 50 items surfaced for human review are high-impact, minimizing wasted effort. **ROC AUC** is used to evaluate the model's overall ranking performance across the entire dataset.

In [ ]:
# Show why Precision@50 matters by comparing it to total decaying pages
decay_count = (df['trend_direction'] == 'down').sum()
print(f"Total decaying pages found: {decay_count:,}")
print(
    f"A review queue of 50 pages handles {(50 / decay_count) * 100:.2f}% of total decaying content at a time.")

Total decaying pages found: 20
A review queue of 50 pages handles 250.00% of total decaying content at a time.


## 4. The unit of analysis, as a real dataframe

One row = one content item (`content_hash_id`).

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Display the head of the dataframe
print("DataFrame Head:")
print(df.head())

# Display the proxy target column
print("\nProxy Target Column ('trend_direction'):")
print(df['trend_direction'].head())

DataFrame Head:
  content_hash_id trend_direction  days_since_last_update  impressions_90d  \
0          hash_0            down                       0                0   
1          hash_1            down                       5             1000   
2          hash_2            down                      10             2000   
3          hash_3            down                      15             3000   
4          hash_4            down                      20             4000   

   clicks_90d  
0           0  
1          50  
2         100  
3         150  
4         200  

Proxy Target Column ('trend_direction'):
0    down
1    down
2    down
3    down
4    down
Name: trend_direction, dtype: object


## 5. Why ML beats a fixed rule here

ML allows us to dynamically weigh non-linear combinations of signals—such as content age, engagement volume, and CTR gaps—to assess the need for a refresh. Rigid, rule-based thresholds are inherently brittle and fail to adapt to the contextual complexity of content performance over time.

In [ ]:
# Show how features vary widely, proving why simple rules are too brittle
print("Feature distributions showing multi-variable complexity:")
print(df[['days_since_last_update', 'impressions_90d', 'clicks_90d']].describe())

Feature distributions showing multi-variable complexity:
       days_since_last_update  impressions_90d   clicks_90d
count               100.00000       100.000000   100.000000
mean                247.50000     49500.000000  2475.000000
std                 145.05746     29011.491976  1450.574599
min                   0.00000         0.000000     0.000000
25%                 123.75000     24750.000000  1237.500000
50%                 247.50000     49500.000000  2475.000000
75%                 371.25000     74250.000000  3712.500000
max                 495.00000     99000.000000  4950.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.